# Upset-Aware Feature Experiment

This notebook tests whether explicitly adding upset-risk features improves the frozen 2010-2022 World Cup backtest.

The experiment is leakage-safe at the tournament level: for each World Cup, all upset-rate features are estimated only from matches before that tournament begins. No new Python source files or generated CSV reports are written.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
YEARS = [2010, 2014, 2018, 2022]
SEED = 20260707
LABELS = {'H': 0, 'D': 1, 'A': 2}
ORDERED = np.array(['H', 'D', 'A'])

matches = pd.read_csv(DATA, parse_dates=['date']).sort_values(['date', 'match_id']).reset_index(drop=True)
assert {'date', 'match_id', 'home_team', 'away_team', 'result', 'tournament', 'neutral', 'home_elo', 'away_elo', 'elo_home_probability', 'elo_draw_probability', 'elo_away_probability'} <= set(matches.columns)
matches.shape


(49493, 22)

In [2]:
def classify_tournament(name):
    text = str(name).lower()
    if 'qualif' in text:
        return 'qualifier'
    if str(name) == 'FIFA World Cup':
        return 'world_cup'
    markers = ['copa america', 'copa américa', 'african cup of nations', 'afc asian cup', 'uefa euro', 'european championship', 'gold cup', 'oceania', 'nations league']
    if any(marker in text for marker in markers):
        return 'continental'
    if text == 'friendly':
        return 'friendly'
    return 'other'

def add_world_cup_stage(frame):
    frame = frame.copy()
    frame['wc_match_number'] = -1
    mask = frame.tournament.eq('FIFA World Cup')
    frame.loc[mask, 'wc_match_number'] = frame.loc[mask].groupby(frame.loc[mask, 'date'].dt.year).cumcount()
    frame['wc_group_stage'] = frame.wc_match_number.between(0, 47).astype(float)
    frame['wc_knockout_stage'] = (frame.wc_match_number >= 48).astype(float)
    return frame

def build_team_panel(frame):
    points = {'H': 1.0, 'D': 0.5, 'A': 0.0}
    home = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.home_team, 'is_home': True,
        'goals_for': frame.home_score.astype(float), 'goals_against': frame.away_score.astype(float),
        'team_elo': frame.home_elo.astype(float), 'result_points': frame.result.map(points).astype(float),
        'expected_points': frame.elo_home_probability + 0.5 * frame.elo_draw_probability,
    })
    away = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.away_team, 'is_home': False,
        'goals_for': frame.away_score.astype(float), 'goals_against': frame.home_score.astype(float),
        'team_elo': frame.away_elo.astype(float), 'result_points': 1.0 - frame.result.map(points).astype(float),
        'expected_points': frame.elo_away_probability + 0.5 * frame.elo_draw_probability,
    })
    panel = pd.concat([home, away], ignore_index=True).sort_values(['team', 'date', 'match_id'], kind='mergesort').reset_index(drop=True)
    grouped = panel.groupby('team', sort=False)
    panel['days_since_match'] = grouped.date.diff().dt.days.fillna(365).clip(0, 365)
    panel['recent_matches_365'] = 0.0
    for _, idx in panel.groupby('team', sort=False).groups.items():
        dates = panel.loc[idx, 'date'].to_numpy()
        counts = []
        for pos, current in enumerate(dates):
            prior = dates[:pos]
            counts.append(float(np.sum((current - prior) <= np.timedelta64(365, 'D'))))
        panel.loc[idx, 'recent_matches_365'] = counts
    for window in [3, 5, 10]:
        for col in ['goals_for', 'goals_against', 'result_points']:
            fill = {'goals_for': 1.25, 'goals_against': 1.25, 'result_points': 0.5}[col]
            panel[f'rolling_{window}_{col}'] = grouped[col].transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()).fillna(fill)
        actual = grouped.result_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        expected = grouped.expected_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        panel[f'rolling_{window}_form_vs_expected'] = (actual - expected).fillna(0.0)
        panel[f'rolling_{window}_elo_trend'] = (grouped.team_elo.shift(1) - grouped.team_elo.shift(1 + window)).fillna(0.0)
    return panel

def build_base_features(frame):
    frame = add_world_cup_stage(frame)
    frame['tournament_type'] = frame.tournament.map(classify_tournament)
    panel = build_team_panel(frame)
    value_cols = [c for c in panel.columns if c.startswith('rolling_')] + ['days_since_match', 'recent_matches_365']
    home = panel.loc[panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'home_{c}' for c in value_cols})
    away = panel.loc[~panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'away_{c}' for c in value_cols})
    out = frame.merge(home, on='match_id', how='left').merge(away, on='match_id', how='left')
    out['abs_elo_difference'] = out.elo_difference.abs()
    out['elo_similarity'] = np.exp(-out.abs_elo_difference / 200.0)
    out['elo_difference_sq'] = np.sign(out.elo_difference) * (out.elo_difference / 400.0) ** 2
    for window in [3, 5, 10]:
        out[f'recent_goal_total_{window}'] = out[f'home_rolling_{window}_goals_for'] + out[f'away_rolling_{window}_goals_for']
        out[f'recent_defence_total_{window}'] = out[f'home_rolling_{window}_goals_against'] + out[f'away_rolling_{window}_goals_against']
        out[f'recent_form_gap_{window}'] = out[f'home_rolling_{window}_result_points'] - out[f'away_rolling_{window}_result_points']
        out[f'recent_attack_gap_{window}'] = out[f'home_rolling_{window}_goals_for'] - out[f'away_rolling_{window}_goals_for']
        out[f'recent_defence_gap_{window}'] = out[f'home_rolling_{window}_goals_against'] - out[f'away_rolling_{window}_goals_against']
    out['rest_gap'] = out.home_days_since_match - out.away_days_since_match
    out['activity_gap_365'] = out.home_recent_matches_365 - out.away_recent_matches_365
    out['elo_favorite'] = np.where(out.elo_home_probability >= out.elo_away_probability, 'H', 'A')
    out['favorite_win_probability'] = np.where(out.elo_favorite == 'H', out.elo_home_probability, out.elo_away_probability)
    out['underdog_win_probability'] = np.where(out.elo_favorite == 'H', out.elo_away_probability, out.elo_home_probability)
    out['favorite_elo'] = np.where(out.elo_favorite == 'H', out.home_elo, out.away_elo)
    out['underdog_elo'] = np.where(out.elo_favorite == 'H', out.away_elo, out.home_elo)
    out['favorite_elo_gap'] = out.favorite_elo - out.underdog_elo
    out['favorite_probability_margin'] = out.favorite_win_probability - out.underdog_win_probability
    out['elo_draw_pressure'] = out.elo_draw_probability / np.maximum(out.favorite_probability_margin, 0.01)
    return out.sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True)

featured = build_base_features(matches)
assert featured.filter(regex='rolling_|days_since|recent_|elo_similarity|stage|gap|pressure|probability').isna().sum().sum() == 0
featured.shape


(49493, 88)

In [3]:
BASE_FEATURES = [
    'elo_difference', 'abs_elo_difference', 'elo_difference_sq', 'elo_similarity', 'neutral', 'home_elo', 'away_elo', 'wc_group_stage', 'wc_knockout_stage',
    'home_days_since_match', 'away_days_since_match', 'home_recent_matches_365', 'away_recent_matches_365', 'rest_gap', 'activity_gap_365',
]
for window in [3, 5, 10]:
    BASE_FEATURES += [
        f'home_rolling_{window}_goals_for', f'home_rolling_{window}_goals_against', f'home_rolling_{window}_result_points', f'home_rolling_{window}_form_vs_expected', f'home_rolling_{window}_elo_trend',
        f'away_rolling_{window}_goals_for', f'away_rolling_{window}_goals_against', f'away_rolling_{window}_result_points', f'away_rolling_{window}_form_vs_expected', f'away_rolling_{window}_elo_trend',
        f'recent_goal_total_{window}', f'recent_defence_total_{window}', f'recent_form_gap_{window}', f'recent_attack_gap_{window}', f'recent_defence_gap_{window}',
    ]

UPSET_FEATURES = [
    'favorite_win_probability', 'underdog_win_probability', 'favorite_probability_margin', 'favorite_elo_gap', 'elo_draw_pressure',
    'elo_gap_bucket_favorite_fail_rate', 'elo_gap_bucket_underdog_win_rate', 'elo_gap_bucket_draw_rate',
    'favorite_team_prior_fail_rate', 'underdog_team_prior_win_rate', 'favorite_team_prior_draw_rate', 'underdog_team_prior_draw_rate',
]
TOURNAMENT_LEVELS = ['world_cup', 'qualifier', 'continental', 'friendly']

def labels_of(frame):
    return frame.result.map(LABELS).to_numpy()

def sample_weights(frame):
    age_years = (frame.date.max() - frame.date).dt.days / 365.25
    recency = np.exp(-age_years / 7.0)
    type_weight = frame.tournament_type.map({'world_cup': 1.7, 'qualifier': 1.25, 'continental': 1.15, 'friendly': 0.65, 'other': 1.0}).fillna(1.0)
    return np.asarray(recency * type_weight, dtype=float)

def add_train_only_upset_rates(train, target):
    train = train.copy()
    target = target.copy()
    train['favorite_failed'] = train.result.ne(train.elo_favorite)
    train['underdog_won'] = ((train.elo_favorite == 'H') & (train.result == 'A')) | ((train.elo_favorite == 'A') & (train.result == 'H'))
    train['draw_vs_favorite'] = train.result.eq('D')
    train['favorite_team'] = np.where(train.elo_favorite == 'H', train.home_team, train.away_team)
    train['underdog_team'] = np.where(train.elo_favorite == 'H', train.away_team, train.home_team)
    target['favorite_team'] = np.where(target.elo_favorite == 'H', target.home_team, target.away_team)
    target['underdog_team'] = np.where(target.elo_favorite == 'H', target.away_team, target.home_team)
    bins = [-np.inf, 50, 100, 150, 200, 300, np.inf]
    train['elo_gap_bucket'] = pd.cut(train.favorite_elo_gap, bins=bins)
    target['elo_gap_bucket'] = pd.cut(target.favorite_elo_gap, bins=bins)
    priors = {
        'elo_gap_bucket_favorite_fail_rate': float(train.favorite_failed.mean()),
        'elo_gap_bucket_underdog_win_rate': float(train.underdog_won.mean()),
        'elo_gap_bucket_draw_rate': float(train.draw_vs_favorite.mean()),
        'favorite_team_prior_fail_rate': float(train.favorite_failed.mean()),
        'underdog_team_prior_win_rate': float(train.underdog_won.mean()),
        'favorite_team_prior_draw_rate': float(train.draw_vs_favorite.mean()),
        'underdog_team_prior_draw_rate': float(train.draw_vs_favorite.mean()),
    }
    maps = {
        'elo_gap_bucket_favorite_fail_rate': train.groupby('elo_gap_bucket', observed=False).favorite_failed.mean(),
        'elo_gap_bucket_underdog_win_rate': train.groupby('elo_gap_bucket', observed=False).underdog_won.mean(),
        'elo_gap_bucket_draw_rate': train.groupby('elo_gap_bucket', observed=False).draw_vs_favorite.mean(),
        'favorite_team_prior_fail_rate': train.groupby('favorite_team').favorite_failed.mean(),
        'underdog_team_prior_win_rate': train.groupby('underdog_team').underdog_won.mean(),
        'favorite_team_prior_draw_rate': train.groupby('favorite_team').draw_vs_favorite.mean(),
        'underdog_team_prior_draw_rate': train.groupby('underdog_team').draw_vs_favorite.mean(),
    }
    key_for = {
        'elo_gap_bucket_favorite_fail_rate': 'elo_gap_bucket',
        'elo_gap_bucket_underdog_win_rate': 'elo_gap_bucket',
        'elo_gap_bucket_draw_rate': 'elo_gap_bucket',
        'favorite_team_prior_fail_rate': 'favorite_team',
        'underdog_team_prior_win_rate': 'underdog_team',
        'favorite_team_prior_draw_rate': 'favorite_team',
        'underdog_team_prior_draw_rate': 'underdog_team',
    }
    for name, mapping in maps.items():
        key = key_for[name]
        target[name] = target[key].map(mapping).astype(float).fillna(priors[name])
        train[name] = train[key].map(mapping).astype(float).fillna(priors[name])
    return train, target

def design(frame, include_upset=False):
    cols = BASE_FEATURES + (UPSET_FEATURES if include_upset else [])
    out = frame[cols].astype(float).copy()
    for col in [c for c in out.columns if 'elo' in c or c in ['home_elo', 'away_elo', 'elo_difference', 'abs_elo_difference', 'favorite_elo_gap']]:
        out[col] = out[col] / 400.0
    for col in ['home_days_since_match', 'away_days_since_match', 'rest_gap']:
        if col in out:
            out[col] = out[col] / 30.0
    out['neutral'] = frame.neutral.astype(float)
    for level in TOURNAMENT_LEVELS:
        out[f'tournament_{level}'] = (frame.tournament_type == level).astype(float)
    return out

def normalize(probs):
    probs = np.clip(np.asarray(probs, dtype=float), 1e-12, None)
    return probs / probs.sum(axis=1, keepdims=True)

def score(labels, probs):
    y = pd.Series(labels).map(LABELS).to_numpy()
    actual = np.eye(3)[y]
    return {
        'log_loss': log_loss(y, probs, labels=[0, 1, 2]),
        'brier': float(np.mean(np.sum((probs - actual) ** 2, axis=1))),
        'accuracy': float((probs.argmax(axis=1) == y).mean()),
        'correct': int((probs.argmax(axis=1) == y).sum()),
    }


In [4]:
rows = []
prediction_rows = []

for year in YEARS:
    test_raw = featured[(featured.tournament == 'FIFA World Cup') & (featured.date.dt.year == year)].copy()
    assert len(test_raw) == 64
    cutoff = test_raw.date.min()
    train_raw = featured[(featured.date < cutoff) & (featured.date >= '2000-01-01')].copy()
    assert train_raw.date.max() < cutoff
    train_upset, test_upset = add_train_only_upset_rates(train_raw, test_raw)
    y_train = labels_of(train_raw)
    y_test = labels_of(test_raw)

    for name, train_frame, test_frame, include_upset in [
        ('weighted_logistic_current_best', train_raw, test_raw, False),
        ('weighted_logistic_upset_aware', train_upset, test_upset, True),
    ]:
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.25, max_iter=3000, random_state=SEED))
        model.fit(design(train_frame, include_upset), y_train, logisticregression__sample_weight=sample_weights(train_frame))
        probs = normalize(model.predict_proba(design(test_frame, include_upset)))
        metric = score(test_frame.result, probs)
        rows.append({'world_cup': year, 'model': name, 'matches': len(test_frame), **metric})
        if name.endswith('upset_aware'):
            for idx, row in enumerate(test_frame.itertuples(index=False)):
                prediction_rows.append({
                    'world_cup': year,
                    'match': f'{row.home_team} vs {row.away_team}',
                    'actual': row.result,
                    'prediction': ORDERED[probs[idx].argmax()],
                    'favorite': row.elo_favorite,
                    'favorite_fail_rate_feature': row.elo_gap_bucket_favorite_fail_rate,
                    'underdog_win_rate_feature': row.elo_gap_bucket_underdog_win_rate,
                })

metrics = pd.DataFrame(rows)
aggregate = metrics.groupby('model').apply(lambda g: pd.Series({
    'log_loss': np.average(g.log_loss, weights=g.matches),
    'brier': np.average(g.brier, weights=g.matches),
    'accuracy': np.average(g.accuracy, weights=g.matches),
    'correct': int(g.correct.sum()),
    'matches': int(g.matches.sum()),
}), include_groups=False).sort_values(['accuracy', 'log_loss'], ascending=[False, True])

display(metrics.pivot(index='world_cup', columns='model', values='accuracy').round(4))
display(aggregate.round(4))

base = aggregate.loc['weighted_logistic_current_best']
upset = aggregate.loc['weighted_logistic_upset_aware']
print(f"Current best: {int(base.correct)}/{int(base.matches)} = {base.accuracy:.2%}")
print(f"Upset-aware: {int(upset.correct)}/{int(upset.matches)} = {upset.accuracy:.2%}")
print(f"Delta: {int(upset.correct - base.correct):+d} matches, {upset.accuracy - base.accuracy:+.2%}")

assert len(metrics) == 8
assert aggregate.matches.eq(256).all()
assert metrics[['log_loss', 'brier', 'accuracy']].apply(np.isfinite).all().all()


model,weighted_logistic_current_best,weighted_logistic_upset_aware
world_cup,,
2010,0.5625,0.5156
2014,0.5625,0.5625
2018,0.6250,0.5938
2022,0.5625,0.5469


,log_loss,brier,accuracy,correct,matches
model,,,,,
weighted_logistic_current_best,0.9732,0.5691,0.5781,148.0,256.0
weighted_logistic_upset_aware,0.9936,0.5808,0.5547,142.0,256.0


Current best: 148/256 = 57.81%
Upset-aware: 142/256 = 55.47%
Delta: -6 matches, -2.34%
